## SQL DATA ANALYSIS with Python in Jupyter Notebook


## Step 1: Install libraries


In [ ]:
# No additional packages are required for SQLite in Python


## Step 2: Connect and reset the SQLite database


In [ ]:
import sqlite3

conn = sqlite3.connect("diabetes.db")
cursor = conn.cursor()
cursor.execute("DROP VIEW IF EXISTS payroll_summary")
cursor.execute("DROP TABLE IF EXISTS pay")
conn.commit()


## Step 3: Create a new table


In [ ]:
cursor.execute("""
CREATE TABLE pay (
    Id INTEGER PRIMARY KEY AUTOINCREMENT,
    Employee_Name TEXT,
    days_worked REAL,
    Hours_worked REAL,
    PER_DAY_SALARY REAL
)
""")
conn.commit()


## Step 4: Insert data


In [ ]:
employees = [
    ("Alex", 5.5, 44.0, 650),
    ("Augustine", 6, 48.0, 1100),
    ("Alois", 6, 48.0, 1500),
    ("Julius", 5, 40.0, 1400),
    ("Vincent", 4, 32.0, 650),
    ("EMMAH", 4.5, 36, 700)
]
cursor.executemany(
    "INSERT INTO pay (Employee_Name, days_worked, Hours_worked, PER_DAY_SALARY) VALUES (?, ?, ?, ?)",
    employees
)
conn.commit()


In [ ]:
for row in cursor.execute("SELECT * FROM pay"):
    print(row)


## Example query


In [ ]:
result = cursor.execute("SELECT AVG(PER_DAY_SALARY) FROM pay").fetchone()
print("Average daily salary:", result[0])


## Step 5: Check data quality

Before calculating payroll, check for missing values, invalid work totals, and duplicate employee names.

In [ ]:
quality_checks = cursor.execute("""
SELECT
    COUNT(*) AS employee_count,
    SUM(CASE WHEN Employee_Name IS NULL OR TRIM(Employee_Name) = '' THEN 1 ELSE 0 END) AS missing_names,
    SUM(CASE WHEN days_worked <= 0 OR Hours_worked <= 0 OR PER_DAY_SALARY <= 0 THEN 1 ELSE 0 END) AS invalid_rows,
    COUNT(DISTINCT Employee_Name) AS unique_names
FROM pay
""").fetchone()
print("Employees:", quality_checks[0])
print("Missing names:", quality_checks[1])
print("Invalid rows:", quality_checks[2])
print("Duplicate names:", quality_checks[0] - quality_checks[3])


## Step 6: Build a payroll summary view

The view centralizes calculations that can be reused in later reports. Total pay is based on days worked and the daily salary, while hourly pay makes compensation easier to compare across employees.

In [ ]:
cursor.execute("DROP VIEW IF EXISTS payroll_summary")
cursor.execute("""
CREATE VIEW payroll_summary AS
SELECT
    Id,
    Employee_Name,
    days_worked,
    Hours_worked,
    PER_DAY_SALARY,
    ROUND(days_worked * PER_DAY_SALARY, 2) AS total_pay,
    ROUND((days_worked * PER_DAY_SALARY) / NULLIF(Hours_worked, 0), 2) AS hourly_pay,
    ROUND(Hours_worked / NULLIF(days_worked, 0), 2) AS hours_per_day
FROM pay
""")
conn.commit()

for row in cursor.execute("SELECT * FROM payroll_summary ORDER BY total_pay DESC"):
    print(row)


## Step 7: Generate a ranked payroll report

In [ ]:
ranked_report = cursor.execute("""
SELECT
    RANK() OVER (ORDER BY total_pay DESC) AS pay_rank,
    Employee_Name,
    total_pay,
    hourly_pay,
    hours_per_day
FROM payroll_summary
ORDER BY pay_rank, Employee_Name
""").fetchall()

print("Rank | Employee | Total pay | Hourly pay | Hours/day")
print("-" * 55)
for row in ranked_report:
    print(f"{row[0]:>4} | {row[1]:<8} | {row[2]:>9.2f} | {row[3]:>10.2f} | {row[4]:>9.2f}")


## Step 8: Compare payroll by work-load band

In [ ]:
workload_summary = cursor.execute("""
SELECT
    CASE
        WHEN days_worked >= 6 THEN 'Full schedule'
        WHEN days_worked >= 5 THEN 'Regular schedule'
        ELSE 'Part schedule'
    END AS workload_band,
    COUNT(*) AS employees,
    ROUND(AVG(days_worked), 2) AS average_days,
    ROUND(AVG(total_pay), 2) AS average_total_pay,
    ROUND(SUM(total_pay), 2) AS payroll_cost
FROM payroll_summary
GROUP BY workload_band
ORDER BY payroll_cost DESC
""").fetchall()

for row in workload_summary:
    print(row)


## Step 9: Model a performance bonus

This scenario adds a 10% bonus to employees who worked at least 48 hours. The original pay table is not changed.

In [ ]:
bonus_report = cursor.execute("""
SELECT
    Employee_Name,
    total_pay,
    CASE WHEN Hours_worked >= 48 THEN ROUND(total_pay * 0.10, 2) ELSE 0 END AS bonus,
    ROUND(total_pay + CASE WHEN Hours_worked >= 48 THEN total_pay * 0.10 ELSE 0 END, 2) AS pay_with_bonus
FROM payroll_summary
ORDER BY pay_with_bonus DESC
""").fetchall()

for row in bonus_report:
    print(row)
print("Total payroll with bonus:", sum(row[3] for row in bonus_report))


## Step 10: Use a parameterized employee search

In [ ]:
minimum_hours = 40
matching_employees = cursor.execute("""
SELECT Employee_Name, Hours_worked, total_pay
FROM payroll_summary
WHERE Hours_worked >= ?
ORDER BY total_pay DESC
""", (minimum_hours,)).fetchall()

print(f"Employees who worked at least {minimum_hours} hours:")
for employee in matching_employees:
    print(employee)


## Step 11: Final business insights

The queries above show total payroll, individual rankings, workload differences, hourly compensation, and the impact of a possible bonus. Keeping these calculations in SQL makes the report easy to extend when more employees are added.

In [ ]:
totals = cursor.execute("""
SELECT
    COUNT(*),
    ROUND(SUM(total_pay), 2),
    ROUND(AVG(hourly_pay), 2),
    MAX(total_pay),
    MIN(total_pay)
FROM payroll_summary
""").fetchone()
print(f"Employees: {totals[0]}")
print(f"Total payroll: {totals[1]:.2f}")
print(f"Average hourly pay: {totals[2]:.2f}")
print(f"Highest employee total: {totals[3]:.2f}")
print(f"Lowest employee total: {totals[4]:.2f}")

conn.close()
